# Byte Pair Encoding (BPE) 簡易實作範例

## 教學目標

- 本教學著重於讓學生理解 BPE 的核心演算法流程
- 透過程式碼直接觀察 subword tokenization 的過程與結果

## 教學內容
- 使用課堂範例作為 corpus，做一個極簡版本的 BPE。

## 1. 為什麼需要 BPE？

如果我們把每個英文單字都當成一個 token，遇到沒看過的新字時就很麻煩，因為新字不存在於 **vocab** 中，我們無法將新字對應到其 embedding。

例如模型的 **vocab** 中有 `low`、`lower`、`lowest`，但是沒有 `newest`。

BPE 的想法是：

1. 先把單字拆成很小的單位（這裡先用字元）
2. 找出最常一起出現的相鄰 pair
3. 把它們 merge 成一個新的 symbol
4. 重複很多次

最後就會得到一些常見的 **subword units**，那麼即使是新的字，也能透過 subwords 取得對應的 embeddings，也就是 out of vocabulary (OOV) 問題。

In [ ]:
# 0. 引入相關套件

from collections import Counter # 統計字詞出現次數

In [ ]:
# 1. 使用課堂範例作為 corpus

words = [
    "low", "low", "low", "low", "low", "lower", "lower", "newest", "newest",
    "newest", "newest", "newest", "newest", "widest", "widest", "widest",
]
print(words)

['low', 'low', 'low', 'low', 'low', 'lower', 'lower', 'newest', 'newest', 'newest', 'newest', 'newest', 'newest', 'widest', 'widest', 'widest']


## 2. 先把每個字拆成字元

為了方便，我們在每個單字後面加上一個 end-of-word 標記 `</w>`。
這樣比較容易看出單字邊界。

In [ ]:
# 2. 使用 list 把每個字 (string) 拆成字元 (characters)

tokens = [list(w) + ["</w>"] for w in words]
tokens

[['l', 'o', 'w', '</w>'],
 ['l', 'o', 'w', '</w>'],
 ['l', 'o', 'w', '</w>'],
 ['l', 'o', 'w', '</w>'],
 ['l', 'o', 'w', '</w>'],
 ['l', 'o', 'w', 'e', 'r', '</w>'],
 ['l', 'o', 'w', 'e', 'r', '</w>'],
 ['n', 'e', 'w', 'e', 's', 't', '</w>'],
 ['n', 'e', 'w', 'e', 's', 't', '</w>'],
 ['n', 'e', 'w', 'e', 's', 't', '</w>'],
 ['n', 'e', 'w', 'e', 's', 't', '</w>'],
 ['n', 'e', 'w', 'e', 's', 't', '</w>'],
 ['n', 'e', 'w', 'e', 's', 't', '</w>'],
 ['w', 'i', 'd', 'e', 's', 't', '</w>'],
 ['w', 'i', 'd', 'e', 's', 't', '</w>'],
 ['w', 'i', 'd', 'e', 's', 't', '</w>']]

### 注意!!! ⚠️
**這裡的 `tokens` 是 characters (字元)**

## 3. get_pair_counts: 統計相鄰 pair

例如 `['l', 'o', 'w', '</w>']` 會產生這些 pair：

- (`l`, `o`)
- (`o`, `w`)
- (`w`, `</w>`)

下面這個函式會把整個 corpus 中所有相鄰 pair 的出現次數統計出來。

In [ ]:
# 3. 統計相鄰字元對 (character pairs) 出現的次數

def get_pair_counts(
    tokenized_words: list[list[str]]
) -> Counter[tuple[str, str]]:
    """
    遍歷目前的語料庫（tokenized_words），計算每一對相鄰 token 出現的總頻率。
    例如：['l', 'o', 'w', '</w>'] 會產生 ('l', 'o'), ('o', 'w'), ('w', '</w>') 三個對。
    """
    counts = Counter()

    for word in tokenized_words:
        # 從左到右一個一個取字元出來
        for i in range(len(word) - 1):
            now = word[i]
            next = word[i + 1]
            pair = (now, next)  # 將相鄰的兩個 token 組合成一個 tuple (元組) 作為鍵
            counts[pair] += 1   # 在計數器中累加該對出現的次數
    return counts

# 呼叫函式並取得目前語料庫中所有 pair 的頻率
pair_counts = get_pair_counts(tokens)

# 因為我們是用 Counter 計算次數，因此我們可以使用.most_common() 的功能
# 這可以幫助我們快速找出出現次數最高的前 10 名，作為合併的候選對象
pair_counts.most_common(10)

[(('e', 's'), 9),
 (('s', 't'), 9),
 (('t', '</w>'), 9),
 (('w', 'e'), 8),
 (('l', 'o'), 7),
 (('o', 'w'), 7),
 (('n', 'e'), 6),
 (('e', 'w'), 6),
 (('w', '</w>'), 5),
 (('w', 'i'), 3)]

## 4. merge_pair: 把最常見的 pair merge 起來

如果最常見 pair 是 `('e', 's')`，我們就把它合成 `'st'`。

也就是：

```
['n', 'e', 'w', 'e', 's', 't']
->
['n', 'e', 'w', 'es', 't']
```

In [ ]:
# 4-1. 定義 function: 給定新字，合併舊的字元對

def merge_pair(
    tokenized_words: list[str],
    pair_to_merge: tuple[str]
) -> list[list[str]]:
    """
    將語料庫中指定的 (a, b) 對合併為 "ab"。
    例如：pair_to_merge 為 ('t', 'h')，則 ['t', 'h', 'e'] 會變成 ['th', 'e']。
    """
    a, b = pair_to_merge
    merged_symbol = a + b  # 建立合併後的新 token (例如 'th')

    new_tokenized_words = []
    for word in tokenized_words:
        new_word = []
        i = 0
        while i < len(word):
            # word[i] == a 代表左邊那個字等於 a
            # word[i + 1] == b 代表右邊那個字等於 b
            # i < len(word) - 1 代表不要觸及到最右邊 (因為一次需要考慮兩格)
            if i < len(word) - 1 and word[i] == a and word[i + 1] == b:
                new_word.append(merged_symbol)  # 將合併後的新符號放入新單字列表
                i += 2 # 有配對到的話一次跳兩格 (跳過原本的 a 和 b)
            else:
                new_word.append(word[i])       # 保持原樣，放入目前的 token
                i += 1 # 沒有配對到的話就換下一格
        new_tokenized_words.append(new_word)    # 儲存處理完畢的單字

    return new_tokenized_words

# 找到現在最常出現的 pair
# .most_common(1) 會回傳 list，例如 [ (('e', 'r'), 50) ]
# 取 [0][0] 是為了拿取該 tuple 對 (即 ('e', 'r'))
best_pair = pair_counts.most_common(1)[0][0]
best_pair

('e', 's')

In [ ]:
# 4-2. 執行 `merge_pair` function: 給定新字，合併舊的字元對

tokens_after_one_merge = merge_pair(tokens, best_pair)
tokens_after_one_merge

[['l', 'o', 'w', '</w>'],
 ['l', 'o', 'w', '</w>'],
 ['l', 'o', 'w', '</w>'],
 ['l', 'o', 'w', '</w>'],
 ['l', 'o', 'w', '</w>'],
 ['l', 'o', 'w', 'e', 'r', '</w>'],
 ['l', 'o', 'w', 'e', 'r', '</w>'],
 ['n', 'e', 'w', 'es', 't', '</w>'],
 ['n', 'e', 'w', 'es', 't', '</w>'],
 ['n', 'e', 'w', 'es', 't', '</w>'],
 ['n', 'e', 'w', 'es', 't', '</w>'],
 ['n', 'e', 'w', 'es', 't', '</w>'],
 ['n', 'e', 'w', 'es', 't', '</w>'],
 ['w', 'i', 'd', 'es', 't', '</w>'],
 ['w', 'i', 'd', 'es', 't', '</w>'],
 ['w', 'i', 'd', 'es', 't', '</w>']]

## 5. 重複做幾次 merge

這就是 BPE 的核心：

- 統計 pair
- 找最常見的
- merge
- 再統計一次
- 再 merge

下面我們做一個簡單版本，每一步都印出來。

### `run_bpe` 中為什麼要多做 [w[:] for w in a]
```python
a = [['l', 'o', 'w'], ['n', 'e', 'w']]
b = [w[:] for w in a]

print(a == b)   # True
print(a is b)   # False
print(a[0] is b[0])  # False
```
### [w for w in a] vs. [w[:] for w in a] 的情況
```python
a = [['l', 'o', 'w'], ['n', 'e', 'w']]

# 情況一：普通的清單生成式
b = [w for w in a]
# 這裡 b 是一個新的 list 物件，但裡面的子 list (例如 ['l', 'o', 'w'])
# 還是指向 a 的同一個記憶體位址。

# 情況二：使用切片 [:] 的生成式
c = [w[:] for w in a]
# 這裡 w[:] 會對每一個子 list 進行「切片複製」。
# 這意味著 c 裡面的子 list 是全新的物件，內容雖然跟 a 一樣，但記憶體位址不同。

print(a is b)        # False (外層 list 不同人)
print(a[0] is b[0])  # True  (內層的單字是同一個人！改了 a[0]，b[0] 也會跟著變)

print(a is c)        # False (外層 list 不同人)
print(a[0] is c[0])  # False (內層的單字也是不同人！這才是真正獨立的備份)
```

### 目前定義好的 functions (將用於 `run_bpe`)
| function name | 輸入 | 輸出 |
| - | - | - |
| `get_pair_counts` | `每筆資料 的 tokens` | `每種 pairs 的統計次數` |
| `merge_pair`| `每筆資料 的 tokens`, `新配對` | `每筆資料的 pairs` |

In [ ]:
# 5. 綜合前面的 functions，執行 BPE
#它將「統計」與「合併」這兩個動作重複執行，直到達到我們設定的合併次數

def run_bpe(tokenized_words: list[str], num_merges: int = 5):
    # 複製一份新的 `tokenized_words`
    # 複製之後，外層 list 是新的，內層也是新的
    # 這是為了避免在函式內修改資料時，意外影響到外部傳進來的原始變數
    tokenized_words = [w[:] for w in tokenized_words]
    merge_rules = [] # 用來記錄合併的順序，這在未來進行推論 (Inference) 時非常重要

    for step in range(1, num_merges + 1):
        # 每一輪都要重新統計當前語料庫中所有 pair 的次數
        pair_counts = get_pair_counts(tokenized_words)
        if not pair_counts:
            # 現在已經沒有任何 pair 可以再 merge (例如剩下一堆單獨字元)，就提前停止。
            break

        # 找出當前頻率最高的字元對
        best_pair, freq = pair_counts.most_common(1)[0]
        # 記錄這條合併規則 (例如: ('e', 'r'))
        merge_rules.append(best_pair)
        # 執行實際的合併動作，產生更新後的語料庫
        tokenized_words = merge_pair(tokenized_words, best_pair)

        # 列印目前的處理進度，方便觀察字元如何漸漸變成 subword
        print(f"Step {step}: merge {best_pair} -> '{best_pair[0] + best_pair[1]}' (freq={freq})")
        for original, new_tokens in zip(words, tokenized_words):
            # :7s 用來排版 (欄位寬度至少 7 個字元，讓輸出整齊對齊)
            print(f"  {original:7s} -> {new_tokens}")
        print()

    return tokenized_words, merge_rules

# 執行 BPE，設定合併 4 次
final_tokens, merge_rules = run_bpe(tokens, num_merges=4)

Step 1: merge ('e', 's') -> 'es' (freq=9)
  low     -> ['l', 'o', 'w', '</w>']
  low     -> ['l', 'o', 'w', '</w>']
  low     -> ['l', 'o', 'w', '</w>']
  low     -> ['l', 'o', 'w', '</w>']
  low     -> ['l', 'o', 'w', '</w>']
  lower   -> ['l', 'o', 'w', 'e', 'r', '</w>']
  lower   -> ['l', 'o', 'w', 'e', 'r', '</w>']
  newest  -> ['n', 'e', 'w', 'es', 't', '</w>']
  newest  -> ['n', 'e', 'w', 'es', 't', '</w>']
  newest  -> ['n', 'e', 'w', 'es', 't', '</w>']
  newest  -> ['n', 'e', 'w', 'es', 't', '</w>']
  newest  -> ['n', 'e', 'w', 'es', 't', '</w>']
  newest  -> ['n', 'e', 'w', 'es', 't', '</w>']
  widest  -> ['w', 'i', 'd', 'es', 't', '</w>']
  widest  -> ['w', 'i', 'd', 'es', 't', '</w>']
  widest  -> ['w', 'i', 'd', 'es', 't', '</w>']

Step 2: merge ('es', 't') -> 'est' (freq=9)
  low     -> ['l', 'o', 'w', '</w>']
  low     -> ['l', 'o', 'w', '</w>']
  low     -> ['l', 'o', 'w', '</w>']
  low     -> ['l', 'o', 'w', '</w>']
  low     -> ['l', 'o', 'w', '</w>']
  lower   -> ['l'

In [ ]:
# 5-2. 什麼時候會 not pair_counts?

print(f"觀察原本的 {tokens}")
print()

# 這裡模擬了一個極端情況：
# 每個單字在 list 裡都已經被合併成「唯一一個」超長 token 了
tmp_tokenized_words = [['low</w>'], ['newer</w>']]

# get_pair_counts 的邏輯是 for i in range(len(word) - 1)
# 當 word 只有一個元素時，len(word) 為 1，則 range(1 - 1) 就是 range(0)
# 這會導致內層迴圈完全不執行，因此回傳空的 Counter
get_pair_counts(tmp_tokenized_words)

觀察原本的 [['l', 'o', 'w', '</w>'], ['l', 'o', 'w', '</w>'], ['l', 'o', 'w', '</w>'], ['l', 'o', 'w', '</w>'], ['l', 'o', 'w', '</w>'], ['l', 'o', 'w', 'e', 'r', '</w>'], ['l', 'o', 'w', 'e', 'r', '</w>'], ['n', 'e', 'w', 'e', 's', 't', '</w>'], ['n', 'e', 'w', 'e', 's', 't', '</w>'], ['n', 'e', 'w', 'e', 's', 't', '</w>'], ['n', 'e', 'w', 'e', 's', 't', '</w>'], ['n', 'e', 'w', 'e', 's', 't', '</w>'], ['n', 'e', 'w', 'e', 's', 't', '</w>'], ['w', 'i', 'd', 'e', 's', 't', '</w>'], ['w', 'i', 'd', 'e', 's', 't', '</w>'], ['w', 'i', 'd', 'e', 's', 't', '</w>']]



Counter()

## 6. 觀察一下結果

請回答：

Q1：哪些單字共享了一些片段？
在 BPE 的過程中，我們會發現具有相同語法功能或相同字根的單字會共享片段。例如：

詞尾 (Suffix)：lowest 和 newest 會共享 est 或 est</w>。

字根 (Stem)：low、lower、lowest 會共享 low 這個片段。

常見組合：如 th、ing、ion 等，幾乎在所有英文語料中都會被共享。

Q2：merge 次數變多後，token 比較像字元，還是比較像整個詞？
Merge 次數少：Token 比較像「字元 (Characters)」。詞彙表很小（只有字母），但一個句子會被切得很碎。

Merge 次數多：Token 會漸漸演變成「完整詞 (Full Words)」。

BPE 的目標：是在兩者之間取得平衡。我們希望常見的詞（如 the）直接變成一個 Token，而罕見的詞（如 unbelievably）則被切成有意義的片段（un + believe + ably）。

Q3：low、lower、lowest 為什麼容易共享 subword？
這是因為 BPE 是基於統計頻率的：

這三個詞都包含了共同的字母組合 l-o-w。

在語料庫中，只要這三個詞出現的總次數夠多，('l', 'o') 和 ('lo', 'w') 的出現頻率就會遠高於其他罕見組合。

因此，演算法會優先把 low 合併在一起。

結果就是：low 變成了一個獨立的 Subword，並同時存在於這三個詞的 Tokenization 結果中。

In [ ]:
merge_rules

[('e', 's'), ('es', 't'), ('est', '</w>'), ('l', 'o')]

## 7. 用學到的 merge rules 來切新字

現在我們拿一個新字 `newest` 來試試看。

注意：這裡是教學版，所以我們只做很直接的規則套用。

In [ ]:
def apply_merges_to_word(
    word: str,
    merge_rules: list[tuple[str]]
):
    """
    將學習到的 BPE 合併規則，應用在一個全新的單字上。
    這就是模型在推論時如何將單字切換成 Subwords 的過程。
    """
    # 任何輸入的字先幫它加上邊界符號 </w>
    # 這樣演算法才能識別單字的結尾，與訓練時的格式一致
    tokens = list(word) + ["</w>"]

    # 必須按照 merge_rules 的「先後順序」依序進行合併
    # 這是因為後面的規則可能依賴於前面規則合併出的結果
    for pair in merge_rules:
        # 這裡呼叫先前定義的 merge_pair 函式
        # 因為 merge_pair 預期輸入是 list of list，所以傳入 [tokens]
        # 回傳後取 [0] 拿回處理好的單一單字 list
        tokens = merge_pair([tokens], pair)[0]

    return tokens

# 測試 1：訓練集中有的單字
test_word = "newest"
print(apply_merges_to_word(test_word, merge_rules))
# 預期：['n', 'e', 'w', 'est</w>']

# 測試 2：基礎詞
test_word = "low"
print(apply_merges_to_word(test_word, merge_rules))
# 預期：['lo', 'w', '</w>']

# 測試 3：從未見過的生字 (Out-of-vocabulary)
test_word = "widest"
print(apply_merges_to_word(test_word, merge_rules))
# 預期：['w', 'i', 'd', 'est</w>']
# 雖然沒看過 wide，但因為看過 est</w>，所以能成功切出有意義的字尾

['n', 'e', 'w', 'est</w>']
['lo', 'w', '</w>']
['w', 'i', 'd', 'est</w>']


## 8. 小練習

你可以試著修改下面幾個地方：

- 把 `num_merges=5` 改成 `2` 或 `10`
- 把 corpus 換成 `run`, `running`, `runner`, `runs`
- 測試新的單字，例如 `lowest`, `newer`, `runningly`

思考：

- merge 次數太少時會怎樣？
- merge 次數太多時又會怎樣？
- BPE 為什麼比把整個單字直接當 token 更有彈性？

# 6. 實驗與觀察：手動調整參數

# 嘗試 1：更換語料庫 (Corpus)
# 這裡使用了具有明顯詞綴 (running, runner) 的單字，適合觀察 -ing, -er, -s 的合併
words = ["run", "running", "runner", "runs"]
tokens = [list(w) + ["</w>"] for w in words]

# 嘗試 2：調整 num_merges
# 修改此處：num_merges=2 (觀察初步合併) 或 num_merges=10 (觀察過度合併)
final_tokens, merge_rules = run_bpe(tokens, num_merges=5)

# 測試新單字
test_words = ["lowest", "newer", "runningly"]
for tw in test_words:
    print(f"Testing '{tw}':", apply_merges_to_word(tw, merge_rules))

1. Merge 次數太少時會怎樣？
現象：Token 會停留在非常細碎的狀態（大部分是單個字母）。

影響：序列長度（Sequence Length）會變得很長，模型處理起來很累；此外，模型很難學到「意義」，因為它看到的都是碎掉的字母，而不是具有語意的片段。

2. Merge 次數太多時又會怎樣？
現象：Token 會傾向於變成「整個單字」。

影響：雖然序列變短了，但詞彙表 (Vocabulary) 會變得巨大。如果次數多到極限，BPE 就退化成了 Word-level tokenization。遇到沒看過的生字時，又會回到 [UNK] (未知詞) 的惡夢。

3. BPE 為什麼比把整個單字直接當 token 更有彈性？
這是 BPE 成為主流 NLP 模型（如 GPT、BERT）核心技術的原因：

處理未見詞 (OOV)：如上面的實驗，如果訓練看過 running 和 runner，即使測試遇到生字 runningly，BPE 也能把它拆解成 run + ning + ly。模型只要認識這些片段，就能猜出這是一個「關於跑的副詞」。

語意共享：run、running、runs 都能共享同一個 run token，這讓模型能更輕易地學到這些詞之間的關聯性。

壓縮效率：它能用較小的詞彙表，表達出幾乎無限的單字組合。

## 9. 總結

這份 code 範例中，可以觀察到：

- BPE 會從小單位開始組合
- 常一起出現的 pair 會被逐步 merge
- 最後形成比較常見、可重用的 subword

這也是為什麼很多 NLP / LLM tokenizer 不直接只用 word-level，而會使用 subword 方法，因為可以避免 word-level 的 OOV 問題。